In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Ensure timestamp is datetime in UTC then convert to Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

for df in (df_metric, df_trace, df_log, df_error):
    # convert integer/float seconds to timezone-aware datetimes in Asia/Shanghai
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) metric.csv aggregation: group by cmdb_id and kpi_name -> count and global P95 of value
metric_aggr = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(count='count')
)

# compute p95 per group (using the full series)
p95_metric = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .quantile(0.95)
    .reset_index(name='p95_value')
)

# merge counts and p95 into metric_aggr (full aggregate stored)
metric_aggr = metric_aggr.merge(p95_metric, on=['cmdb_id', 'kpi_name'])

# Prepare compact display (up to 50 rows sorted by count desc)
metric_display = metric_aggr.sort_values('count', ascending=False).head(50)[['cmdb_id', 'kpi_name', 'count', 'p95_value']]

# 2) trace.csv aggregation: group by cmdb_id and trace_name -> count and global P95
trace_aggr = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(count='count')
)
p95_trace = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .quantile(0.95)
    .reset_index(name='p95_value')
)
trace_aggr = trace_aggr.merge(p95_trace, on=['cmdb_id', 'trace_name'])
trace_display = trace_aggr.sort_values('count', ascending=False).head(50)[['cmdb_id', 'trace_name', 'count', 'p95_value']]

# 3) log.csv aggregation: group by cmdb_id and log_name -> count and global P95
log_aggr = (
    df_log
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .agg(count='count')
)
p95_log = (
    df_log
    .groupby(['cmdb_id', 'log_name'])['value']
    .quantile(0.95)
    .reset_index(name='p95_value')
)
log_aggr = log_aggr.merge(p95_log, on=['cmdb_id', 'log_name'])
log_display = log_aggr.sort_values('count', ascending=False).head(50)[['cmdb_id', 'log_name', 'count', 'p95_value']]

# 4) error_logs.csv aggregation: group by cmdb_id -> count, earliest ts, latest ts, sample message (first)
# If 'message' column exists, use it; else create empty.
if 'message' not in df_error.columns:
    df_error['message'] = ""

err_grp = df_error.sort_values('timestamp').groupby('cmdb_id')
error_aggr = pd.DataFrame({
    'cmdb_id': err_grp.size().index,
    'count': err_grp.size().values,
    'earliest_timestamp': err_grp['timestamp'].min().values,
    'latest_timestamp': err_grp['timestamp'].max().values,
    'sample_message': err_grp['message'].first().values
})
# Keep full error_aggr in kernel variable; for display keep compact
error_aggr_display = error_aggr[['cmdb_id', 'count', 'earliest_timestamp', 'latest_timestamp', 'sample_message']]

# Final compact outputs (up to requested limits). These variables are left in kernel for later steps.
metric_display, trace_display, log_display, error_aggr_display

```
Out[1]:
```
```python
summary = (
    "Aggregations completed and saved to metric_aggr, trace_aggr, log_aggr, error_aggr. "
    "Key findings:\n"
    "- metric: many KPI groups contain 31 samples; Tomcat04 appears frequently among top-count KPIs. "
    "Notable metric: IG01 JVM-Threads (count=31) has a global P95 ~74.5 (high thread count). "
    "- trace: dockerB2 shows the highest-count trace groups (count=31) with sizable P95s "
    "(e.g., to_MG02.row_count p95~167.5, duration_p95~13.90). IG01 self trace duration_mean p95~2.47. "
    "- logs: Tomcat02 and Tomcat01 have the largest log.row_count groups (p95 ~614–632); log.error_count p95 is 0 for many components. "
    "Mysql log groups have far fewer points (count=30) and much lower row_count p95. "
    "- error_logs: no error log rows were found (error_aggr is empty).\n"
    "If you want more detail, I can show the top rows from any of metric_aggr, trace_aggr, log_aggr, or error_aggr."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count  p95_value
1575  Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31        0.0
1539  Tomcat04     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31        0.0
6         IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31       74.5
1550  Tomcat04      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31        1.0
1554  Tomcat04         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31        3.0
...        ...                                                ...    ...        ...
1207  Tomcat01    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31        0.0
1197  Tomcat01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  7769928.0
1186  Tomcat01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31        1.0
1330  Tomcat02     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31        0.0
1331  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31        0.0

[50 rows x 4 columns],       cmdb_id                         trace_name  count   p95_value
263  dockerB2            trace.to_MG02.row_count     31  167.500000
262  dockerB2         trace.to_MG02.duration_p95     31   13.895150
261  dockerB2        trace.to_MG02.duration_mean     31    3.603309
260  dockerB2            trace.to_MG01.row_count     31  152.500000
3        IG01           trace.self.duration_mean     31    2.474770
..        ...                                ...    ...         ...
50       MG01      trace.from_dockerA1.row_count     31   86.500000
37       MG01   trace.from_Tomcat01.duration_p95     31   13.415475
38       MG01      trace.from_Tomcat01.row_count     31  310.000000
39       MG01  trace.from_Tomcat02.duration_mean     31    2.572382
40       MG01   trace.from_Tomcat02.duration_p95     31   16.695200

[50 rows x 4 columns],      cmdb_id          log_name  count  p95_value
9   Tomcat02     log.row_count     31   614.5000
7   Tomcat02   log.error_count     31     0.0000
6   Tomcat01     log.row_count     31   632.5000
4   Tomcat01   log.error_count     31     0.0000
18  apache02   log.error_count     31     0.0000
17  apache01     log.row_count     31  1221.5000
19  apache02     log.row_count     31  1215.5000
16  apache01   log.error_count     31     0.0000
10  Tomcat03   log.error_count     31     0.0000
15  Tomcat04     log.row_count     31   615.0000
13  Tomcat04   log.error_count     31     0.0000
12  Tomcat03     log.row_count     31   610.0000
1    Mysql01     log.row_count     30    48.6500
0    Mysql01   log.error_count     30     0.0000
14  Tomcat04  log.gc_pause_max     27     0.0970
11  Tomcat03  log.gc_pause_max     25     0.1000
8   Tomcat02  log.gc_pause_max     24     0.1085
5   Tomcat01  log.gc_pause_max     24     0.1185
3    Mysql02     log.row_count      2     1.0000
2    Mysql02   log.error_count      2     0.0000, Empty DataFrame
Columns: [cmdb_id, count, earliest_timestamp, latest_timestamp, sample_message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_metric if available; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 per cmdb_id and kpi_name using the full series (thresholds computed BEFORE filtering)
metric_thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(global_p95=lambda s: s.quantile(0.95),
         global_p5=lambda s: s.quantile(0.05))
    .reset_index()
)

# 2) Filter metric.csv to incident window: March 24, 2021, 15:00 to 15:30 (UTC+8)
start_ts = pd.Timestamp("2021-03-24 15:00:00").tz_localize(tz)
end_ts = pd.Timestamp("2021-03-24 15:30:00").tz_localize(tz)

metric_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# 3) Join filtered rows with thresholds and identify anomalies (value >= P95 OR value <= P5)
metric_window = metric_window.merge(metric_thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomalies
metric_window['is_anomaly'] = (
    (metric_window['value'] >= metric_window['global_p95']) |
    (metric_window['value'] <= metric_window['global_p5'])
)

# Save full anomaly rows for later steps
metric_anomaly_rows = metric_window[metric_window['is_anomaly']].copy()

# 4) Aggregate anomalies by cmdb_id and kpi_name
if not metric_anomaly_rows.empty:
    ag = metric_anomaly_rows.groupby(['cmdb_id', 'kpi_name']).agg(
        anomaly_count=('value', 'count'),
        earliest_timestamp=('timestamp', 'min'),
        latest_timestamp=('timestamp', 'max'),
        min_value=('value', 'min'),
        max_value=('value', 'max'),
        global_p95=('global_p95', 'first'),
        global_p5=('global_p5', 'first')
    ).reset_index()

    # severity_score = max_value / global_p95 when global_p95>0 else large number
    ag['severity_score'] = np.where(
        ag['global_p95'] > 0,
        ag['max_value'] / ag['global_p95'],
        1e9
    )

    # 5) Prepare final compact table: top 20 rows sorted by anomaly_count desc then severity_score desc
    metric_anomalies = ag.sort_values(['anomaly_count', 'severity_score'], ascending=[False, False]) \
                         .head(20)[['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_timestamp',
                                    'latest_timestamp', 'min_value', 'max_value',
                                    'global_p5', 'global_p95', 'severity_score']]
else:
    # No anomalies found in window: create empty DataFrame with target columns
    metric_anomalies = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_timestamp',
        'latest_timestamp', 'min_value', 'max_value', 'global_p5',
        'global_p95', 'severity_score'
    ])

# Keep metric_thresholds and metric_anomaly_rows in kernel for later steps
metric_thresholds, metric_anomaly_rows.shape, metric_anomalies

```
Out[2]:
```
```python
summary = (
    "Summary of metric anomaly detection for 2021-03-24 15:00–15:30 (UTC+8):\n"
    "- Thresholds (global P95/P5) were computed for each cmdb_id+kpi using the full series and saved to metric_thresholds.\n"
    "- The incident-window filter produced 19,379 anomaly rows (metric_anomaly_rows) where value >= P95 or <= P5.\n"
    "- Aggregated anomalies (metric_anomalies) show many groups with anomaly_count = 31 (full 31-minute window), "
    "notably many Tomcat session KPIs and OSLinux TCP state metrics across Tomcat01, Tomcat02, MG02, etc.\n"
    "- A large subset of anomalous groups have global_p95 == 0 and global_p5 == 0, so min/max values are 0 and "
    "severity_score was set to a very large number (placeholder) — this indicates zero-valued series or zero thresholds and "
    "makes the severity metric artificially large.\n"
    "- Next steps I recommend: inspect metric_anomaly_rows for representative groups to confirm whether zero values are valid "
    "or indicate collection/normalization issues; cross-check with logs/traces and error logs for those components."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name    global_p95     global_p5
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09
1         IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  4.077978e+01  5.035845e+00
2         IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.385209e+08  1.056938e+08
3         IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.848308e+07  5.848308e+07
4         IG01          JVM-Operating System_7778_JVM_JVM_CPULoad  6.441350e-01  4.762850e-01
...        ...                                                ...           ...           ...
1761  dockerA2  Container-DOCKER_CONTAINER_76d31070b844--bcou-...  0.000000e+00  0.000000e+00
1762  dockerA2  Container-DOCKER_CONTAINER_76d31070b844--bcou-...  0.000000e+00  0.000000e+00
1763  dockerA2  Container-DOCKER_CONTAINER_76d31070b844--bcou-...  0.000000e+00  0.000000e+00
1764  dockerA2  Container-DOCKER_CONTAINER_76d31070b844--bcou-...  0.000000e+00  0.000000e+00
1765  dockerA2  Container-DOCKER_CONTAINER_76d31070b844--bcou-...  0.000000e+00  0.000000e+00

[1766 rows x 4 columns], (19379, 7),        cmdb_id                                           kpi_name  anomaly_count        earliest_timestamp          latest_timestamp  min_value  max_value  global_p5  global_p95  severity_score
363       MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09
364       MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09
1175  Tomcat01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09
1207  Tomcat01    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09
1208  Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09
1209  Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09
1210  Tomcat01        Tomcat-Sessions_7441--_SESSIONActiveCounter             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09
1211  Tomcat01     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09
1212  Tomcat01     Tomcat-Sessions_7441--_SESSIONRejectedSessions             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09
1213  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09
1214  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09
1215  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09
1293  Tomcat02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09
1325  Tomcat02    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09
1326  Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09
1327  Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09
1328  Tomcat02        Tomcat-Sessions_7441--_SESSIONActiveCounter             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09
1329  Tomcat02     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09
1330  Tomcat02     Tomcat-Sessions_7441--_SESSIONRejectedSessions             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09
1331  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00        0.0        0.0        0.0         0.0    1.000000e+09)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_anomalies from kernel; if not present, raise informative error
try:
    metric_anomalies
except NameError:
    raise NameError("metric_anomalies not found in kernel. Please run the anomaly detection step first.")

# 1) Filter out rows where both global_p95 == 0 AND global_p5 == 0
kpi_anom_nonzero = metric_anomalies[~((metric_anomalies['global_p95'] == 0) & (metric_anomalies['global_p5'] == 0))].copy()

# Ensure timestamp columns are timezone-aware in UTC+8
for col in ['earliest_timestamp', 'latest_timestamp']:
    if col in kpi_anom_nonzero.columns and not pd.api.types.is_datetime64tz_dtype(kpi_anom_nonzero[col]):
        kpi_anom_nonzero[col] = pd.to_datetime(kpi_anom_nonzero[col]).dt.tz_localize(tz)

# 2) Group remaining rows by cmdb_id and compute required aggregates
if not kpi_anom_nonzero.empty:
    grp = kpi_anom_nonzero.groupby('cmdb_id', as_index=False)

    comp_summary = grp.agg(
        num_kpis_with_anomalies=('kpi_name', 'nunique'),
        total_anomaly_points=('anomaly_count', 'sum'),
        earliest_anomaly=('earliest_timestamp', 'min'),
        latest_anomaly=('latest_timestamp', 'max'),
        max_severity_score=('severity_score', 'max')
    )

    # Determine top_kpi and top_kpi_count per cmdb_id
    # Use idxmax on anomaly_count within each group
    idx = kpi_anom_nonzero.groupby('cmdb_id')['anomaly_count'].idxmax()
    top_kpis = kpi_anom_nonzero.loc[idx, ['cmdb_id', 'kpi_name', 'anomaly_count']].rename(
        columns={'kpi_name': 'top_kpi', 'anomaly_count': 'top_kpi_count'}
    )

    # Merge top_kpis into comp_summary
    component_anom_summary = comp_summary.merge(top_kpis, on='cmdb_id', how='left')

    # 3) Sort by total_anomaly_points desc then max_severity_score desc and keep top 20
    component_anom_summary = component_anom_summary.sort_values(['total_anomaly_points', 'max_severity_score'],
                                                                ascending=[False, False]).head(20)

else:
    # No KPI-level anomalies with non-zero thresholds
    component_anom_summary = pd.DataFrame(columns=[
        'cmdb_id', 'num_kpis_with_anomalies', 'total_anomaly_points',
        'earliest_anomaly', 'latest_anomaly', 'max_severity_score',
        'top_kpi', 'top_kpi_count'
    ])

# Keep kpi_anom_nonzero and component_anom_summary in kernel for later inspection
component_anom_summary

```
Out[3]:
```
summary = (
    "Result summary:\n"
    "- After filtering out KPI groups whose global P95 and P5 are both zero, no KPI-level anomalies remain (component_anom_summary is empty).\n"
    "- This means that although many anomaly points were detected in the incident window, nearly all belonged to KPIs with zero-valued thresholds (global_p95==0 and global_p5==0) and were removed by the filter.\n"
    "- Likely causes: those series are constant zeros or thresholds computed as zero due to normalization/collection, which makes the severity metric meaningless for them.\n"
    "- Recommended next steps: inspect the full anomaly rows (metric_anomaly_rows) to see which KPIs/components had zero thresholds, verify metric collection/normalization for those KPIs, and cross-check relevant components (e.g., Tomcat01/02, MG02) with logs and traces."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, num_kpis_with_anomalies, total_anomaly_points, earliest_anomaly, latest_anomaly, max_severity_score, top_kpi, top_kpi_count]
Index: []```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_trace if available; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure timestamp column is timezone-aware in UTC+8
if not pd.api.types.is_datetime64tz_dtype(df_trace['timestamp']):
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P5 per cmdb_id and trace_name using full series (thresholds BEFORE filtering)
trace_p95 = df_trace.groupby(['cmdb_id', 'trace_name'])['value'].quantile(0.95).reset_index(name='global_p95')
trace_p5  = df_trace.groupby(['cmdb_id', 'trace_name'])['value'].quantile(0.05).reset_index(name='global_p5')

trace_thresholds = trace_p95.merge(trace_p5, on=['cmdb_id', 'trace_name'])

# 2) Filter trace.csv to incident window (2021-03-24 15:00 to 15:30 UTC+8)
start_ts = pd.Timestamp("2021-03-24 15:00:00").tz_localize(tz)
end_ts   = pd.Timestamp("2021-03-24 15:30:00").tz_localize(tz)

trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# 3) Join filtered rows with thresholds and identify anomalies (value >= global_p95 OR value <= global_p5)
trace_window = trace_window.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')

trace_window['is_anomaly'] = (
    (trace_window['value'] >= trace_window['global_p95']) |
    (trace_window['value'] <= trace_window['global_p5'])
)

# Save full anomaly rows for later inspection
trace_anomaly_rows = trace_window[trace_window['is_anomaly']].copy()

# 4) Aggregate anomalies by cmdb_id and trace_name
if not trace_anomaly_rows.empty:
    ag = trace_anomaly_rows.groupby(['cmdb_id', 'trace_name']).agg(
        anomaly_count=('value', 'count'),
        earliest_timestamp=('timestamp', 'min'),
        latest_timestamp=('timestamp', 'max'),
        min_value=('value', 'min'),
        max_value=('value', 'max'),
        global_p5=('global_p5', 'first'),
        global_p95=('global_p95', 'first')
    ).reset_index()

    # severity_score = max_value / global_p95 when global_p95>0 else NaN
    ag['severity_score'] = np.where(ag['global_p95'] > 0, ag['max_value'] / ag['global_p95'], np.nan)

    # 5) Prepare compact table: top 20 rows sorted by anomaly_count desc then severity_score desc
    trace_anomalies = ag.sort_values(['anomaly_count', 'severity_score'], ascending=[False, False]) \
                      .head(20)[['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_timestamp',
                                 'latest_timestamp', 'min_value', 'max_value',
                                 'global_p5', 'global_p95', 'severity_score']]
else:
    trace_anomalies = pd.DataFrame(columns=[
        'cmdb_id', 'trace_name', 'anomaly_count', 'earliest_timestamp',
        'latest_timestamp', 'min_value', 'max_value', 'global_p5',
        'global_p95', 'severity_score'
    ])

# Return compact outputs and keep full data in kernel
trace_thresholds.head(20), trace_anomalies, trace_anomaly_rows.shape

```
Out[4]:
```
```python
summary = (
    "Trace anomaly detection summary for 2021-03-24 15:00–15:30 (UTC+8):\n"
    "- Global P95/P5 thresholds were computed for each cmdb_id+trace_name and saved to trace_thresholds.\n"
    "- The incident-window filter produced 969 anomaly rows (trace_anomaly_rows) where values >= P95 or <= P5.\n"
    "- Top aggregated findings (examples):\n"
    "  • dockerA2 trace.self.duration_p95: 9 anomalies, severity ~7.77 (largest relative spike in duration_p95).\n"
    "  • MG02 ↔ dockerB2 row_count traces: MG02.trace.from_dockerB2 and dockerB2.trace.to_MG02 each 5 anomalies, "
    "max value 240 vs global_p95 167.5 (severity ~1.43) indicating row-count bursts.\n"
    "  • IG01 / Tomcat02 traces: trace.to_Tomcat02.row_count and Tomcat02.trace.from_IG01.row_count have 5 anomalies, "
    "max ~323 vs p95 ~317 (severity ~1.02) — moderate bursts.\n"
    "  • Several duration_p95/duration_mean anomalies across MG01, dockerA1, dockerA2, dockerB2 with anomaly counts 4 and "
    "severity scores between ~2.4–4.3, indicating noticeable latency spikes on those call paths.\n"
    "- Interpretation: the data shows simultaneous row-count spikes (traffic bursts) and latency spikes on paths among MG, dockerA*, dockerB2, and dockerA2. "
    "dockerA2.trace.self.duration_p95 is a prominent outlier.\n"
    "- Recommended next steps: inspect full trace_anomaly_rows for these top groups, correlate with logs/metrics for dockerA2, MG02, dockerB2, IG01, Tomcat02, "
    "and MG01/dockerA1 to find root cause (traffic surge, downstream slowness, or resource contention)."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                       trace_name   global_p95    global_p5
0     IG01    trace.from_root.duration_mean     8.359000     8.359000
1     IG01     trace.from_root.duration_p95    10.830400    10.830400
2     IG01        trace.from_root.row_count     2.000000     2.000000
3     IG01         trace.self.duration_mean     2.474770     0.327338
4     IG01          trace.self.duration_p95    10.469500     1.397625
5     IG01             trace.self.row_count  2520.000000  1636.000000
6     IG01  trace.to_Tomcat01.duration_mean     2.238598     0.350656
7     IG01   trace.to_Tomcat01.duration_p95     9.774325     1.091000
8     IG01      trace.to_Tomcat01.row_count   334.000000   198.500000
9     IG01  trace.to_Tomcat02.duration_mean     2.501364     0.282141
10    IG01   trace.to_Tomcat02.duration_p95    10.555400     1.109675
11    IG01      trace.to_Tomcat02.row_count   317.000000    64.500000
12    IG01  trace.to_Tomcat03.duration_mean     2.711714     0.318941
13    IG01   trace.to_Tomcat03.duration_p95    10.523075     1.106925
14    IG01      trace.to_Tomcat03.row_count   315.500000    60.500000
15    IG01  trace.to_Tomcat04.duration_mean     2.248642     0.316121
16    IG01   trace.to_Tomcat04.duration_p95    10.015400     1.096475
17    IG01      trace.to_Tomcat04.row_count   318.500000   195.500000
18    IG02    trace.from_root.duration_mean     1.702000     1.702000
19    IG02     trace.from_root.duration_p95     3.218500     3.218500,       cmdb_id                         trace_name  anomaly_count        earliest_timestamp          latest_timestamp  min_value   max_value  global_p5  global_p95  severity_score
220  dockerA2            trace.self.duration_p95              9 2021-03-24 15:01:00+08:00 2021-03-24 15:15:00+08:00   0.002000    0.129800   0.002000    0.016700        7.772455
101      MG02      trace.from_dockerB2.row_count              5 2021-03-24 15:06:00+08:00 2021-03-24 15:30:00+08:00   3.000000  240.000000  38.000000  167.500000        1.432836
263  dockerB2            trace.to_MG02.row_count              5 2021-03-24 15:06:00+08:00 2021-03-24 15:30:00+08:00   3.000000  240.000000  38.000000  167.500000        1.432836
11       IG01        trace.to_Tomcat02.row_count              5 2021-03-24 15:05:00+08:00 2021-03-24 15:30:00+08:00   3.000000  323.000000  64.500000  317.000000        1.018927
140  Tomcat02          trace.from_IG01.row_count              5 2021-03-24 15:05:00+08:00 2021-03-24 15:30:00+08:00   3.000000  323.000000  64.500000  317.000000        1.018927
70       MG01     trace.to_dockerA2.duration_p95              4 2021-03-24 15:01:00+08:00 2021-03-24 15:30:00+08:00   0.745500   11.206700   1.024675    2.584725        4.335742
211  dockerA2       trace.from_MG01.duration_p95              4 2021-03-24 15:01:00+08:00 2021-03-24 15:30:00+08:00   0.745500   11.206700   1.024675    2.584725        4.335742
109      MG02     trace.to_dockerA1.duration_p95              4 2021-03-24 15:01:00+08:00 2021-03-24 15:30:00+08:00   0.747000   11.227300   0.952925    3.133975        3.582447
196  dockerA1       trace.from_MG02.duration_p95              4 2021-03-24 15:01:00+08:00 2021-03-24 15:30:00+08:00   0.747000   11.227300   0.952925    3.133975        3.582447
69       MG01    trace.to_dockerA2.duration_mean              4 2021-03-24 15:08:00+08:00 2021-03-24 15:14:00+08:00   0.117294    2.257237   0.121762    0.696169        3.242367
210  dockerA2      trace.from_MG01.duration_mean              4 2021-03-24 15:08:00+08:00 2021-03-24 15:14:00+08:00   0.117294    2.257237   0.121762    0.696169        3.242367
90       MG02  trace.from_dockerA1.duration_mean              4 2021-03-24 15:06:00+08:00 2021-03-24 15:30:00+08:00   0.006000   12.122500   0.011590    4.140223        2.927982
207  dockerA1        trace.to_MG02.duration_mean              4 2021-03-24 15:06:00+08:00 2021-03-24 15:30:00+08:00   0.006000   12.122500   0.011590    4.140223        2.927982
219  dockerA2           trace.self.duration_mean              4 2021-03-24 15:01:00+08:00 2021-03-24 15:17:00+08:00   0.014533    0.286830   0.015779    0.098222        2.920236
66       MG01    trace.to_dockerA1.duration_mean              4 2021-03-24 15:01:00+08:00 2021-03-24 15:30:00+08:00   0.023250    2.386948   0.122538    0.879119        2.715161
192  dockerA1      trace.from_MG01.duration_mean              4 2021-03-24 15:01:00+08:00 2021-03-24 15:30:00+08:00   0.023250    2.386948   0.122538    0.879119        2.715161
256  dockerB2            trace.self.duration_p95              4 2021-03-24 15:08:00+08:00 2021-03-24 15:24:00+08:00   0.013000    4.648000   0.029850    1.731550        2.684300
67       MG01     trace.to_dockerA1.duration_p95              4 2021-03-24 15:01:00+08:00 2021-03-24 15:30:00+08:00   0.026550   12.521050   1.028125    4.721175        2.652105
193  dockerA1       trace.from_MG01.duration_p95              4 2021-03-24 15:01:00+08:00 2021-03-24 15:30:00+08:00   0.026550   12.521050   1.028125    4.721175        2.652105
112      MG02     trace.to_dockerA2.duration_p95              4 2021-03-24 15:01:00+08:00 2021-03-24 15:18:00+08:00   0.949700   11.885300   1.024300    4.863200        2.443926, (969, 7))```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure trace_anomaly_rows exists
try:
    trace_anomaly_rows
except NameError:
    raise NameError("trace_anomaly_rows not found in kernel. Please run trace anomaly detection first.")

# 1) Aggregate trace_anomaly_rows per (cmdb_id, trace_name) to get anomaly_count and other stats
trace_anom_aggr = (
    trace_anomaly_rows
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        anomaly_count=('value', 'count'),
        earliest_timestamp=('timestamp', 'min'),
        latest_timestamp=('timestamp', 'max'),
        min_value=('value', 'min'),
        max_value=('value', 'max'),
        global_p5=('global_p5', 'first'),
        global_p95=('global_p95', 'first')
    )
)

# compute severity_score = max_value / global_p95 when global_p95>0 else NaN
import numpy as np
trace_anom_aggr['severity_score'] = np.where(
    trace_anom_aggr['global_p95'] > 0,
    trace_anom_aggr['max_value'] / trace_anom_aggr['global_p95'],
    np.nan
)

# 1) Per-component summary: group by cmdb_id
# top_trace: trace_name with largest anomaly_count per cmdb_id
idx_top = trace_anom_aggr.groupby('cmdb_id')['anomaly_count'].idxmax()
top_traces = trace_anom_aggr.loc[idx_top, ['cmdb_id', 'trace_name', 'anomaly_count']].rename(
    columns={'trace_name': 'top_trace', 'anomaly_count': 'top_trace_count'}
)

comp_grp = trace_anom_aggr.groupby('cmdb_id', as_index=False).agg(
    earliest_anomaly=('earliest_timestamp', 'min'),
    anomaly_rows_count=('anomaly_count', 'sum'),
    max_severity=('severity_score', 'max')
)

trace_component_summary = comp_grp.merge(top_traces, on='cmdb_id', how='left')

# Sort by earliest_anomaly ascending and keep top 20
trace_component_summary = trace_component_summary.sort_values('earliest_anomaly', ascending=True).head(20)

# 2) Detailed aggregated anomalies for cmdb_id == 'dockerA2', limited to top 20 rows sorted by earliest_timestamp
trace_dockerA2_details = trace_anom_aggr[trace_anom_aggr['cmdb_id'] == 'dockerA2'] \
    .sort_values('earliest_timestamp', ascending=True) \
    .head(20)[['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_timestamp', 'latest_timestamp',
               'min_value', 'max_value', 'global_p5', 'global_p95', 'severity_score']]

# Keep trace_anom_aggr (all aggregated per-trace anomalies), trace_component_summary, and trace_dockerA2_details in kernel
trace_anom_aggr.shape, trace_component_summary, trace_dockerA2_details

```
Out[5]:
```
summary = (
    "Trace anomaly summary (2021-03-24 15:00–15:30 UTC+8):\n"
    "- There are 264 distinct trace-level anomaly groups detected in the incident window.\n"
    "- By component, MG02 and MG01 show the largest anomaly volumes (163 and 162 anomaly rows respectively), "
    "with MG02's top trace being trace.from_dockerB2.row_count and MG01 showing high severity on traces from Tomcat01.\n"
    "- dockerA2 stands out for severity: it has 68 anomaly rows and the highest individual severity score (trace.self.duration_p95, severity ~7.77), "
    "indicating a strong relative spike in p95 latency for dockerA2.\n"
    "- Other notable patterns: row-count bursts between MG nodes and dockerB2/dockerA2 (several row_count traces with anomaly_count=4–5 and severity >1), "
    "and multiple duration_p95/duration_mean spikes across MG, dockerA1/A2/B2 and Tomcat nodes (severity scores ~2–4 in several traces).\n"
    "- Some trace groups have zero thresholds (leading to NaN severities); where thresholds are nonzero, many anomalies reflect real traffic/latency spikes rather than single outliers.\n"
    "- Recommended next steps: focus investigation on dockerA2 (duration_p95 and row_count anomalies) and MG01/MG02 (row_count and duration anomalies), "
    "and correlate these trace anomalies with metrics and logs for those components to identify root cause (traffic surge, downstream slowness, or resource contention)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

((264, 10),      cmdb_id          earliest_anomaly  anomaly_rows_count  max_severity                          top_trace  top_trace_count
3       MG02 2021-03-24 15:00:00+08:00                 163      3.582447      trace.from_dockerB2.row_count                5
4   Tomcat01 2021-03-24 15:00:00+08:00                  66      2.057825      trace.from_IG01.duration_mean                4
5   Tomcat02 2021-03-24 15:00:00+08:00                  64      1.944100          trace.from_IG01.row_count                5
6   Tomcat03 2021-03-24 15:00:00+08:00                  63      1.881655      trace.from_IG01.duration_mean                4
7   Tomcat04 2021-03-24 15:00:00+08:00                  66      1.400233      trace.from_IG01.duration_mean                4
9   dockerA2 2021-03-24 15:00:00+08:00                  68      7.772455            trace.self.duration_p95                9
10  dockerB1 2021-03-24 15:00:00+08:00                  63      2.253671      trace.from_MG01.duration_mean                4
11  dockerB2 2021-03-24 15:00:00+08:00                  64      2.684300            trace.to_MG02.row_count                5
0       IG01 2021-03-24 15:01:00+08:00                  64      1.675900        trace.to_Tomcat02.row_count                5
2       MG01 2021-03-24 15:01:00+08:00                 162      4.335742  trace.from_Tomcat01.duration_mean                4
8   dockerA1 2021-03-24 15:01:00+08:00                  63      3.582447      trace.from_MG01.duration_mean                4
1       IG02 2021-03-24 15:04:00+08:00                  63      1.399665           trace.self.duration_mean                4,       cmdb_id                     trace_name  anomaly_count        earliest_timestamp          latest_timestamp  min_value    max_value    global_p5   global_p95  severity_score
218  dockerA2      trace.from_root.row_count              1 2021-03-24 15:00:00+08:00 2021-03-24 15:00:00+08:00   2.000000     2.000000     2.000000     2.000000        1.000000
216  dockerA2  trace.from_root.duration_mean              1 2021-03-24 15:00:00+08:00 2021-03-24 15:00:00+08:00   0.000000     0.000000     0.000000     0.000000             NaN
217  dockerA2   trace.from_root.duration_p95              1 2021-03-24 15:00:00+08:00 2021-03-24 15:00:00+08:00   0.000000     0.000000     0.000000     0.000000             NaN
211  dockerA2   trace.from_MG01.duration_p95              4 2021-03-24 15:01:00+08:00 2021-03-24 15:30:00+08:00   0.745500    11.206700     1.024675     2.584725        4.335742
213  dockerA2  trace.from_MG02.duration_mean              4 2021-03-24 15:01:00+08:00 2021-03-24 15:17:00+08:00   0.092161     1.924615     0.116459     1.096653        1.754990
214  dockerA2   trace.from_MG02.duration_p95              4 2021-03-24 15:01:00+08:00 2021-03-24 15:18:00+08:00   0.949700    11.885300     1.024300     4.863200        2.443926
219  dockerA2       trace.self.duration_mean              4 2021-03-24 15:01:00+08:00 2021-03-24 15:17:00+08:00   0.014533     0.286830     0.015779     0.098222        2.920236
220  dockerA2        trace.self.duration_p95              9 2021-03-24 15:01:00+08:00 2021-03-24 15:15:00+08:00   0.002000     0.129800     0.002000     0.016700        7.772455
221  dockerA2           trace.self.row_count              4 2021-03-24 15:02:00+08:00 2021-03-24 15:30:00+08:00  56.000000  4909.000000  1805.500000  3340.500000        1.469540
212  dockerA2      trace.from_MG01.row_count              4 2021-03-24 15:06:00+08:00 2021-03-24 15:30:00+08:00   7.000000   349.000000    44.500000   239.500000        1.457203
215  dockerA2      trace.from_MG02.row_count              4 2021-03-24 15:06:00+08:00 2021-03-24 15:30:00+08:00   1.000000   326.000000   128.000000   284.000000        1.147887
224  dockerA2        trace.to_MG01.row_count              4 2021-03-24 15:06:00+08:00 2021-03-24 15:29:00+08:00   5.000000   107.000000     7.250000    89.150000        1.200224
222  dockerA2    trace.to_MG01.duration_mean              4 2021-03-24 15:06:00+08:00 2021-03-24 15:18:00+08:00   0.011400     9.614419     0.017503     5.537396        1.736271
223  dockerA2     trace.to_MG01.duration_p95              4 2021-03-24 15:06:00+08:00 2021-03-24 15:24:00+08:00   0.021000    20.928200     0.027415    16.381517        1.277550
227  dockerA2        trace.to_MG02.row_count              4 2021-03-24 15:07:00+08:00 2021-03-24 15:30:00+08:00   1.000000   120.000000    11.500000   100.500000        1.194030
225  dockerA2    trace.to_MG02.duration_mean              4 2021-03-24 15:08:00+08:00 2021-03-24 15:30:00+08:00   0.012000     9.307667     0.018299     4.188326        2.222288
226  dockerA2     trace.to_MG02.duration_p95              4 2021-03-24 15:08:00+08:00 2021-03-24 15:30:00+08:00   0.012000    26.182500     0.023600    12.855675        2.036649
210  dockerA2  trace.from_MG01.duration_mean              4 2021-03-24 15:08:00+08:00 2021-03-24 15:14:00+08:00   0.117294     2.257237     0.121762     0.696169        3.242367)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_log if available, else load
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
    # convert timestamp to timezone-aware Asia/Shanghai
    df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure timestamp column is timezone-aware in UTC+8
if not pd.api.types.is_datetime64tz_dtype(df_log['timestamp']):
    df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P5 per cmdb_id + log_name using full series
log_p95 = df_log.groupby(['cmdb_id', 'log_name'])['value'].quantile(0.95).reset_index(name='global_p95')
log_p5  = df_log.groupby(['cmdb_id', 'log_name'])['value'].quantile(0.05).reset_index(name='global_p5')
log_thresholds = log_p95.merge(log_p5, on=['cmdb_id', 'log_name'])

# 2) Filter log.csv to incident window: 2021-03-24 15:00 to 15:30 (UTC+8)
start_ts = pd.Timestamp("2021-03-24 15:00:00").tz_localize(tz)
end_ts   = pd.Timestamp("2021-03-24 15:30:00").tz_localize(tz)
log_window = df_log[(df_log['timestamp'] >= start_ts) & (df_log['timestamp'] <= end_ts)].copy()

# 3) Join filtered rows with thresholds and mark anomalies where value >= p95 OR value <= p5
log_window = log_window.merge(log_thresholds, on=['cmdb_id', 'log_name'], how='left')

log_window['is_anomaly'] = (log_window['value'] >= log_window['global_p95']) | (log_window['value'] <= log_window['global_p5'])

# Save full anomaly rows
log_anomaly_rows = log_window[log_window['is_anomaly']].copy()

# 4) Aggregate anomalies by cmdb_id and log_name
if not log_anomaly_rows.empty:
    ag = log_anomaly_rows.groupby(['cmdb_id', 'log_name']).agg(
        anomaly_count=('value', 'count'),
        earliest_timestamp=('timestamp', 'min'),
        latest_timestamp=('timestamp', 'max'),
        min_value=('value', 'min'),
        max_value=('value', 'max'),
        global_p5=('global_p5', 'first'),
        global_p95=('global_p95', 'first')
    ).reset_index()

    ag['severity_score'] = np.where(ag['global_p95'] > 0, ag['max_value'] / ag['global_p95'], np.nan)

    # 5) Return compact table of top 20 aggregated log anomalies sorted by earliest_timestamp asc then anomaly_count desc
    focus_components = {'dockerA2', 'MG01', 'MG02', 'dockerB2', 'IG01', 'Tomcat02', 'Tomcat01'}

    # Prefer showing focused components first; if none, fall back to global top 20
    ag_focus = ag[ag['cmdb_id'].isin(focus_components)].copy()
    if ag_focus.empty:
        log_anomalies = ag.sort_values(['earliest_timestamp', 'anomaly_count'], ascending=[True, False]).head(20)
    else:
        log_anomalies = ag_focus.sort_values(['earliest_timestamp', 'anomaly_count'], ascending=[True, False]).head(20)

    # Keep columns in requested order
    log_anomalies = log_anomalies[['cmdb_id', 'log_name', 'anomaly_count', 'earliest_timestamp',
                                   'latest_timestamp', 'min_value', 'max_value', 'global_p5', 'global_p95', 'severity_score']]
else:
    log_anomalies = pd.DataFrame(columns=[
        'cmdb_id', 'log_name', 'anomaly_count', 'earliest_timestamp',
        'latest_timestamp', 'min_value', 'max_value', 'global_p5', 'global_p95', 'severity_score'
    ])

# 6) Detailed anomaly rows for cmdb_id == 'dockerA2' limited to 20 rows sorted by earliest_timestamp asc
dockerA2_log_details = pd.DataFrame()
if not log_anomaly_rows.empty:
    dockerA2_log_details = (
        log_anomaly_rows[log_anomaly_rows['cmdb_id'] == 'dockerA2']
        .sort_values('timestamp', ascending=True)
        .head(20)
        .loc[:, ['cmdb_id', 'log_name', 'value', 'timestamp', 'global_p5', 'global_p95']]
        .rename(columns={'value': 'value_at_anomaly', 'timestamp': 'timestamp'})
    )

# Keep log_thresholds, log_anomaly_rows, log_anomalies in kernel for later steps
# Return compact summaries (head/selected cols) as final output
log_thresholds.head(20), log_anomalies, dockerA2_log_details.head(20)

```
Out[6]:
```
Summary of log anomaly detection (2021-03-24 15:00–15:30 UTC+8):

- Thresholds computed (global P95/P5) for each cmdb_id+log_name and saved to log_thresholds (examples: Tomcat01 log.row_count P95~632.5 / P5~408; Tomcat01 log.gc_pause_max P95~0.1185 / P5~0.033).

- Aggregated anomalies (log_anomalies) — focused components (Tomcat01, Tomcat02) show the most relevant events:
  - Tomcat01.log.error_count: anomaly_count=31 across the entire window, but global_p95 and global_p5 are 0 → these are zero-valued series/thresholds and not meaningful for severity.
  - Tomcat02.log.error_count: same situation (31 anomalies, zero thresholds).
  - Tomcat01.log.gc_pause_max: anomaly_count=4 (15:02–15:11), min=0.02, max=0.12, global_p95=0.1185, severity ≈ 1.01 → small GC pause spike reaching ~P95.
  - Tomcat01.log.row_count: anomaly_count=4 (15:05–15:30), min=9, max=699, global_p95≈632.5, severity ≈ 1.11 → row-count burst exceeding P95.
  - Tomcat02.log.row_count: anomaly_count=4 (15:05–15:30), min=11, max=665, global_p95≈614.5, severity ≈ 1.08 → similar row-count burst.
  - Tomcat02.log.gc_pause_max: anomaly_count=4 (15:20–15:29), min=0.03, max=0.14, global_p95=0.1085, severity ≈ 1.29 → noticeable GC pause spikes.

- dockerA2: no log anomalies found in the window (dockerA2 detailed anomaly rows empty).

Interpretation and recommended next steps:
- Ignore anomaly flags tied to zero thresholds (log.error_count series) — they likely reflect constant-zero series or collection/normalization issues.
- Prioritize investigation of Tomcat01 and Tomcat02 for the incident window:
  - Correlate the row_count bursts and GC pause spikes with trace/metric anomalies (we already saw trace and metric anomalies around the same time).
  - Check application load (traffic surge), upstream requests, and GC behavior/configuration on those Tomcat hosts.
  - Review resource usage (CPU, memory, I/O) and relevant error logs around the noted timestamps (15:02–15:30).
- If you want, I can show the full log_anomaly_rows for any component or correlate these log anomalies with the trace/metric anomalies already saved.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          log_name  global_p95  global_p5
0    Mysql01   log.error_count      0.0000     0.0000
1    Mysql01     log.row_count     48.6500     3.4500
2    Mysql02   log.error_count      0.0000     0.0000
3    Mysql02     log.row_count      1.0000     1.0000
4   Tomcat01   log.error_count      0.0000     0.0000
5   Tomcat01  log.gc_pause_max      0.1185     0.0330
6   Tomcat01     log.row_count    632.5000   408.0000
7   Tomcat02   log.error_count      0.0000     0.0000
8   Tomcat02  log.gc_pause_max      0.1085     0.0315
9   Tomcat02     log.row_count    614.5000   384.0000
10  Tomcat03   log.error_count      0.0000     0.0000
11  Tomcat03  log.gc_pause_max      0.1000     0.0320
12  Tomcat03     log.row_count    610.0000   402.5000
13  Tomcat04   log.error_count      0.0000     0.0000
14  Tomcat04  log.gc_pause_max      0.0970     0.0330
15  Tomcat04     log.row_count    615.0000   405.0000
16  apache01   log.error_count      0.0000     0.0000
17  apache01     log.row_count   1221.5000   837.0000
18  apache02   log.error_count      0.0000     0.0000
19  apache02     log.row_count   1215.5000   829.0000,     cmdb_id          log_name  anomaly_count        earliest_timestamp          latest_timestamp  min_value  max_value  global_p5  global_p95  severity_score
4  Tomcat01   log.error_count             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00       0.00       0.00     0.0000      0.0000             NaN
7  Tomcat02   log.error_count             31 2021-03-24 15:00:00+08:00 2021-03-24 15:30:00+08:00       0.00       0.00     0.0000      0.0000             NaN
5  Tomcat01  log.gc_pause_max              4 2021-03-24 15:02:00+08:00 2021-03-24 15:11:00+08:00       0.02       0.12     0.0330      0.1185        1.012658
6  Tomcat01     log.row_count              4 2021-03-24 15:05:00+08:00 2021-03-24 15:30:00+08:00       9.00     699.00   408.0000    632.5000        1.105138
9  Tomcat02     log.row_count              4 2021-03-24 15:05:00+08:00 2021-03-24 15:30:00+08:00      11.00     665.00   384.0000    614.5000        1.082181
8  Tomcat02  log.gc_pause_max              4 2021-03-24 15:20:00+08:00 2021-03-24 15:29:00+08:00       0.03       0.14     0.0315      0.1085        1.290323, Empty DataFrame
Columns: [cmdb_id, log_name, value_at_anomaly, timestamp, global_p5, global_p95]
Index: [])```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure required variables exist
for var in ('trace_anomaly_rows', 'log_anomaly_rows', 'metric_anomaly_rows', 'metric_anomalies'):
    if var not in globals():
        raise NameError(f"{var} not found in kernel. Please run prior anomaly detection steps.")

components = ['dockerA2', 'MG01', 'MG02', 'dockerB2', 'IG01', 'Tomcat02', 'Tomcat01']

# Aggregate trace anomalies per trace_name (from trace_anomaly_rows) to get earliest and severity per trace group
trace_aggr_from_rows = (
    trace_anomaly_rows
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        anomaly_count=('value', 'count'),
        trace_earliest_ts=('timestamp', 'min'),
        trace_latest_ts=('timestamp', 'max'),
        trace_max_value=('value', 'max'),
        global_p95=('global_p95', 'first')
    )
)
trace_aggr_from_rows['trace_severity'] = np.where(
    trace_aggr_from_rows['global_p95'] > 0,
    trace_aggr_from_rows['trace_max_value'] / trace_aggr_from_rows['global_p95'],
    np.nan
)

# Aggregate log anomalies per log_name (from log_anomaly_rows)
log_aggr_from_rows = (
    log_anomaly_rows
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        anomaly_count=('value', 'count'),
        log_earliest_ts=('timestamp', 'min'),
        log_latest_ts=('timestamp', 'max'),
        log_max_value=('value', 'max'),
        global_p95=('global_p95', 'first')
    )
)
log_aggr_from_rows['log_severity'] = np.where(
    log_aggr_from_rows['global_p95'] > 0,
    log_aggr_from_rows['log_max_value'] / log_aggr_from_rows['global_p95'],
    np.nan
)

# For metrics use the already-aggregated metric_anomalies (per cmdb_id + kpi_name)
# metric_anomalies has earliest_timestamp, anomaly_count, max_value, global_p95, severity_score
# Normalize column names for merging
metric_aggr = metric_anomalies.copy()
metric_aggr = metric_aggr.rename(columns={
    'earliest_timestamp': 'metric_earliest_ts',
    'latest_timestamp': 'metric_latest_ts',
    'min_value': 'metric_min_value',
    'max_value': 'metric_max_value',
    'global_p5': 'metric_global_p5',
    'global_p95': 'metric_global_p95',
    'severity_score': 'metric_severity',
    'kpi_name': 'metric_name'
})

# Helper to get earliest item per component from an aggregated source
def earliest_per_component(df, comp_col_ts, name_col, severity_col, comp_list, source_prefix):
    """
    df: aggregated DataFrame with 'cmdb_id', timestamp column, name column, severity column
    comp_col_ts: timestamp column name (e.g., 'trace_earliest_ts')
    name_col: item name column (e.g., 'trace_name' or 'log_name' or 'metric_name')
    severity_col: severity column name
    """
    rows = []
    for comp in comp_list:
        sub = df[df['cmdb_id'] == comp]
        if sub.empty:
            rows.append({
                'cmdb_id': comp,
                f'{source_prefix}_earliest_ts': pd.NaT,
                f'{source_prefix}_name': None,
                f'{source_prefix}_severity': np.nan
            })
        else:
            idx = sub[comp_col_ts].idxmin()
            sel = sub.loc[idx]
            rows.append({
                'cmdb_id': comp,
                f'{source_prefix}_earliest_ts': sel[comp_col_ts],
                f'{source_prefix}_name': sel[name_col],
                f'{source_prefix}_severity': sel[severity_col]
            })
    return pd.DataFrame(rows)

# Prepare trace, log, metric earliest rows per component
trace_earliest_df = earliest_per_component(trace_aggr_from_rows, 'trace_earliest_ts', 'trace_name', 'trace_severity', components, 'trace')
log_earliest_df = earliest_per_component(log_aggr_from_rows, 'log_earliest_ts', 'log_name', 'log_severity', components, 'log')
metric_earliest_df = earliest_per_component(metric_aggr, 'metric_earliest_ts', 'metric_name', 'metric_severity', components, 'metric')

# Merge per-component records
comp_df = trace_earliest_df.merge(log_earliest_df, on='cmdb_id').merge(metric_earliest_df, on='cmdb_id')

# Compute overall_earliest and which source produced it
def pick_overall_earliest(row):
    # collect candidates
    candidates = []
    for src in ('trace', 'log', 'metric'):
        ts = row.get(f'{src}_earliest_ts')
        if pd.notna(ts):
            candidates.append((ts, src))
    if not candidates:
        return pd.NaT, None
    # pick minimum timestamp
    candidates.sort(key=lambda x: x[0])
    return candidates[0]

overall = comp_df.apply(lambda r: pick_overall_earliest(r), axis=1)
comp_df['overall_earliest'] = [t for t, s in overall]
comp_df['overall_earliest_source'] = [s for t, s in overall]

# component_max_severity: max across trace_severity, log_severity, metric_severity (ignoring NaN)
comp_df['component_max_severity'] = comp_df[['trace_severity', 'log_severity', 'metric_severity']].max(axis=1, skipna=True)

# Reorder columns as requested
component_earliest_summary = comp_df[[
    'cmdb_id', 'overall_earliest', 'overall_earliest_source',
    'trace_earliest_ts', 'trace_name', 'trace_severity',
    'log_earliest_ts', 'log_name', 'log_severity',
    'metric_earliest_ts', 'metric_name', 'metric_severity',
    'component_max_severity'
]]

# Sort by overall_earliest ascending and limit to 20 rows
component_earliest_summary = component_earliest_summary.sort_values('overall_earliest', na_position='last').head(20)

# Display the compact summary
component_earliest_summary

```
Out[7]:
```
Summary of earliest-anomaly findings for the candidate components (based on saved anomaly tables):

- Earliest anomalies by component (overall earliest timestamp and source):
  - dockerA2 — overall earliest: 2021-03-24 15:00 (source: trace)
  - MG02 — overall earliest: 2021-03-24 15:00 (source: trace)
  - dockerB2 — overall earliest: 2021-03-24 15:00 (source: trace)
  - Tomcat02 — overall earliest: 2021-03-24 15:00 (source: trace)
  - Tomcat01 — overall earliest: 2021-03-24 15:00 (source: trace)
  - MG01 — overall earliest: 2021-03-24 15:01 (source: trace)
  - IG01 — overall earliest: 2021-03-24 15:01 (source: trace)

- Which source produced the earliest anomalies:
  - For all listed components, the earliest flagged anomalies came from traces (trace data). Logs and metrics either appear at the same minute for some components or later, but trace was the first observable source in this window.

- Notable severity observations:
  - MG01 shows a non-artificial max severity ≈ 2.72 (from trace anomalies) — a meaningful elevated severity.
  - IG01 shows a modest trace severity ≈ 1.03.
  - Many metric anomalies have extremely large severity values in the aggregates (1e9) because their global P95 was zero; those are artifacts of zero thresholds and should be treated as unreliable.
  - dockerA2 had early trace anomalies at 15:00 and also previously observed high relative spikes (e.g., trace.self.duration_p95 around 15:01) in earlier analysis — this component warrants attention even if the very earliest trace group had NaN severity.

- Practical interpretation and next steps:
  - Multiple components (dockerA2, MG02, dockerB2, Tomcat01, Tomcat02) first showed trace-level anomalies right at 15:00 — this suggests the incident began with distributed trace/latency or traffic changes rather than a single isolated host.
  - Prioritize investigation of trace anomalies starting at 15:00 on: dockerA2, MG02, dockerB2, Tomcat01, Tomcat02. Also prioritize MG01 because it has the highest non-artifactual severity among the group.
  - Ignore or de-prioritize anomaly flags tied to zero global thresholds (those create artificial severity); instead use trace severity and non-zero log/metric thresholds to rank impact.
  - Correlate these earliest trace anomalies with the metric and log anomaly rows (around 15:00–15:05) for the same components to localize root cause (traffic burst, downstream slowdown, or resource contention).

If you want, I can now: (a) show the specific trace anomaly rows around 15:00 for the top components (dockerA2, MG01, MG02), or (b) correlate those earliest trace timestamps with metric and log rows to produce a timeline for each component. Which would you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id          overall_earliest overall_earliest_source         trace_earliest_ts                       trace_name  trace_severity           log_earliest_ts         log_name  log_severity        metric_earliest_ts                                     metric_name  metric_severity  component_max_severity
0  dockerA2 2021-03-24 15:00:00+08:00                   trace 2021-03-24 15:00:00+08:00    trace.from_root.duration_mean             NaN                       NaT             None           NaN                       NaT                                            None              NaN                     NaN
2      MG02 2021-03-24 15:00:00+08:00                   trace 2021-03-24 15:00:00+08:00    trace.from_Tomcat01.row_count        1.156785                       NaT             None           NaN 2021-03-24 15:00:00+08:00  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     1.000000e+09            1.000000e+09
3  dockerB2 2021-03-24 15:00:00+08:00                   trace 2021-03-24 15:00:00+08:00    trace.from_root.duration_mean        1.000000                       NaT             None           NaN                       NaT                                            None              NaN            1.000000e+00
5  Tomcat02 2021-03-24 15:00:00+08:00                   trace 2021-03-24 15:00:00+08:00    trace.from_root.duration_mean             NaN 2021-03-24 15:00:00+08:00  log.error_count           NaN 2021-03-24 15:00:00+08:00  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     1.000000e+09            1.000000e+09
6  Tomcat01 2021-03-24 15:00:00+08:00                   trace 2021-03-24 15:00:00+08:00    trace.from_root.duration_mean        1.048543 2021-03-24 15:00:00+08:00  log.error_count           NaN 2021-03-24 15:00:00+08:00  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     1.000000e+09            1.000000e+09
1      MG01 2021-03-24 15:01:00+08:00                   trace 2021-03-24 15:01:00+08:00  trace.to_dockerA1.duration_mean        2.715161                       NaT             None           NaN                       NaT                                            None              NaN            2.715161e+00
4      IG01 2021-03-24 15:01:00+08:00                   trace 2021-03-24 15:01:00+08:00   trace.to_Tomcat04.duration_p95        1.032460                       NaT             None           NaN                       NaT                                            None              NaN            1.032460e+00```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_metric if available; otherwise load and convert timestamps
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure timestamp column is timezone-aware in UTC+8
if not pd.api.types.is_datetime64tz_dtype(df_metric['timestamp']):
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define incident window (inclusive)
start_ts = pd.Timestamp("2021-03-24 15:00:00").tz_localize(tz)
end_ts   = pd.Timestamp("2021-03-24 15:30:00").tz_localize(tz)

# Filter rows for dockerA2 in the incident window
dockerA2_metric_rows = df_metric[
    (df_metric['cmdb_id'] == 'dockerA2') &
    (df_metric['timestamp'] >= start_ts) &
    (df_metric['timestamp'] <= end_ts)
].copy()

# Aggregate by kpi_name: count, min(value), max(value), mean(value)
dockerA2_metric_aggr = (
    dockerA2_metric_rows
    .groupby('kpi_name', as_index=False)['value']
    .agg(count='count', min_value='min', max_value='max', mean_value='mean')
    .sort_values('max_value', ascending=False)
    .reset_index(drop=True)
)

# Keep outputs compact: show shape of filtered rows and up to 50 aggregated KPI rows
dockerA2_metric_rows.shape, dockerA2_metric_aggr.head(50)

```
Out[8]:
```
```python
# Summarize dockerA2 metric aggregation results in plain English (compact)
try:
    total_rows = dockerA2_metric_rows.shape[0]
    num_kpis = dockerA2_metric_aggr.shape[0]
    top3 = dockerA2_metric_aggr.head(3).copy()
    num_zero_kpis = int((dockerA2_metric_aggr['max_value'] == 0).sum())
    
    def fmt_val(v):
        return f"{v:,.3g}" if pd.notna(v) else "NA"
    
    top_lines = []
    for _, r in top3.iterrows():
        top_lines.append(f"- {r['kpi_name'][:80]}... (count={int(r['count'])}, max={fmt_val(r['max_value'])}, mean={fmt_val(r['mean_value'])})")
    
    summary = (
        f"DockerA2 metric window (2021-03-24 15:00–15:30 UTC+8):\n"
        f"- Total metric rows in window: {total_rows}\n"
        f"- Distinct KPIs: {num_kpis}\n"
        f"- Top KPIs by max(value):\n" + "\n".join(top_lines) + "\n"
        f"- KPIs with all-zero values (max==0): {num_zero_kpis} of {num_kpis}\n"
        f"- Observation: many KPIs are container-related; a few KPIs show very large constant values (~1e9–3e9) "
        f"while many others are zero. Investigate the top non-zero container KPIs (listed above) for resource/metric meaning "
        f"and verify whether zero-valued series reflect valid emptiness or collection/normalization issues."
    )
except Exception as e:
    summary = f"Could not produce summary: {e}"

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((144, 4),                                              kpi_name  count     min_value     max_value    mean_value
0   Container-DOCKER_CONTAINER_2c2336e2994f--bcou-...      3  3.072000e+09  3.072000e+09  3.072000e+09
1   Container-DOCKER_CONTAINER_2c2336e2994f--bcou-...     10  1.977958e+09  1.983508e+09  1.980701e+09
2   Container-DOCKER_CONTAINER_2c2336e2994f--bcou-...     10  1.285224e+06  3.197472e+06  2.015719e+06
3   Container-DOCKER_CONTAINER_2c2336e2994f--bcou-...     10  2.636090e+05  4.702310e+05  3.559610e+05
4   Container-DOCKER_CONTAINER_2c2336e2994f--bcou-...      5  5.990000e+01  5.990000e+01  5.990000e+01
5   Container-DOCKER_CONTAINER_2c2336e2994f--bcou-...      5  4.000000e-01  8.200000e-01  6.240000e-01
6   Container-DOCKER_CONTAINER_3bf443a64876--bcou-...     13  0.000000e+00  0.000000e+00  0.000000e+00
7   Container-DOCKER_CONTAINER_3bf443a64876--bcou-...      2  0.000000e+00  0.000000e+00  0.000000e+00
8   Container-DOCKER_CONTAINER_3bf443a64876--bcou-...      4  0.000000e+00  0.000000e+00  0.000000e+00
9   Container-DOCKER_CONTAINER_3bf443a64876--bcou-...      7  0.000000e+00  0.000000e+00  0.000000e+00
10  Container-DOCKER_CONTAINER_3bf443a64876--bcou-...      6  0.000000e+00  0.000000e+00  0.000000e+00
11  Container-DOCKER_CONTAINER_69b53a78b2eb--bcou-...      7  0.000000e+00  0.000000e+00  0.000000e+00
12  Container-DOCKER_CONTAINER_69b53a78b2eb--bcou-...      7  0.000000e+00  0.000000e+00  0.000000e+00
13  Container-DOCKER_CONTAINER_69b53a78b2eb--bcou-...      1  0.000000e+00  0.000000e+00  0.000000e+00
14  Container-DOCKER_CONTAINER_69b53a78b2eb--bcou-...      1  0.000000e+00  0.000000e+00  0.000000e+00
15  Container-DOCKER_CONTAINER_76d31070b844--bcou-...      4  0.000000e+00  0.000000e+00  0.000000e+00
16  Container-DOCKER_CONTAINER_76d31070b844--bcou-...      3  0.000000e+00  0.000000e+00  0.000000e+00
17  Container-DOCKER_CONTAINER_76d31070b844--bcou-...     12  0.000000e+00  0.000000e+00  0.000000e+00
18  Container-DOCKER_CONTAINER_76d31070b844--bcou-...     18  0.000000e+00  0.000000e+00  0.000000e+00
19  Container-DOCKER_CONTAINER_76d31070b844--bcou-...     16  0.000000e+00  0.000000e+00  0.000000e+00)```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Incident window
start_ts = pd.Timestamp("2021-03-24 15:00:00").tz_localize(tz)
end_ts   = pd.Timestamp("2021-03-24 15:30:00").tz_localize(tz)

# Target filter
comp = 'dockerA2'
tname = 'trace.self.duration_p95'

# Prefer using existing trace_anomaly_rows if available and contains required columns
use_existing = ('trace_anomaly_rows' in globals()) and ('trace_name' in trace_anomaly_rows.columns)

if use_existing:
    # Ensure timestamp tz-aware
    if not pd.api.types.is_datetime64tz_dtype(trace_anomaly_rows['timestamp']):
        trace_anomaly_rows['timestamp'] = pd.to_datetime(trace_anomaly_rows['timestamp']).dt.tz_localize(tz)
    candidate = trace_anomaly_rows[
        (trace_anomaly_rows['cmdb_id'] == comp) &
        (trace_anomaly_rows['trace_name'] == tname) &
        (trace_anomaly_rows['timestamp'] >= start_ts) &
        (trace_anomaly_rows['timestamp'] <= end_ts)
    ].copy()
    # Ensure global_p5/global_p95 exist; if not try to merge from trace_thresholds
    if ('global_p95' not in candidate.columns or 'global_p5' not in candidate.columns) and ('trace_thresholds' in globals()):
        candidate = candidate.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')
else:
    # Fallback: load df_trace and trace_thresholds if needed, then compute anomalies for the window
    try:
        df_trace
    except NameError:
        df_trace = pd.read_csv("trace.csv")
    # Ensure timestamp tz-aware
    if not pd.api.types.is_datetime64tz_dtype(df_trace['timestamp']):
        df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # Ensure thresholds exist
    if 'trace_thresholds' not in globals():
        trace_p95 = df_trace.groupby(['cmdb_id', 'trace_name'])['value'].quantile(0.95).reset_index(name='global_p95')
        trace_p5  = df_trace.groupby(['cmdb_id', 'trace_name'])['value'].quantile(0.05).reset_index(name='global_p5')
        trace_thresholds = trace_p95.merge(trace_p5, on=['cmdb_id', 'trace_name'])
    # Filter window + target and join thresholds
    candidate = df_trace[
        (df_trace['cmdb_id'] == comp) &
        (df_trace['trace_name'] == tname) &
        (df_trace['timestamp'] >= start_ts) &
        (df_trace['timestamp'] <= end_ts)
    ].merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left').copy()
    # Mark anomalies per definition (value >= p95 OR value <= p5) and keep only anomalies
    candidate['is_anomaly'] = (candidate['value'] >= candidate['global_p95']) | (candidate['value'] <= candidate['global_p5'])
    candidate = candidate[candidate['is_anomaly']].copy()

# Compute severity_score = value / global_p95 when global_p95 > 0 else NaN
candidate['severity_score'] = np.where(candidate['global_p95'] > 0, candidate['value'] / candidate['global_p95'], np.nan)

# Select and sort columns, limit to 50 rows
dockerA2_self_p95_anom = candidate[['timestamp', 'value', 'global_p5', 'global_p95', 'severity_score']].sort_values('timestamp', ascending=True).head(50).reset_index(drop=True)

# Display the result variable (keeps it in kernel for later steps)
dockerA2_self_p95_anom

```
Out[9]:
```
summary = (
    f"DockerA2 trace.self.duration_p95 anomalies within 2021-03-24 15:00–15:30 (UTC+8):\n"
    f"- Total anomaly rows found: {len(dockerA2_self_p95_anom)}.\n"
    f"- Largest spike: at {dockerA2_self_p95_anom.loc[0,'timestamp']} value={dockerA2_self_p95_anom.loc[0,'value']}s, "
    f"global_p95={dockerA2_self_p95_anom.loc[0,'global_p95']}s, severity≈{dockerA2_self_p95_anom.loc[0,'severity_score']:.3f} (value / p95 ≈7.77).\n"
    f"- Another notable exceedance: at 2021-03-24 15:08:00+08:00 value=0.0204s, severity≈1.22 (slightly above P95).\n"
    f"- Most other anomaly points are low-valued (0.002s) equal to the global P5, producing low severity (≈0.12).\n"
    f"- Interpretation: a short, large p95 spike occurred at 15:01 (primary event), with a smaller p95 exceedance at 15:08 and otherwise low-value readings hitting P5. "
    f"Recommend investigating dockerA2 around 15:01 (trace/span details, container resource metrics, and correlated logs) to find root cause of the p95 latency spike."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

timestamp   value  global_p5  global_p95  severity_score
0 2021-03-24 15:01:00+08:00  0.1298      0.002      0.0167        7.772455
1 2021-03-24 15:03:00+08:00  0.0020      0.002      0.0167        0.119760
2 2021-03-24 15:04:00+08:00  0.0020      0.002      0.0167        0.119760
3 2021-03-24 15:06:00+08:00  0.0020      0.002      0.0167        0.119760
4 2021-03-24 15:08:00+08:00  0.0204      0.002      0.0167        1.221557
5 2021-03-24 15:09:00+08:00  0.0020      0.002      0.0167        0.119760
6 2021-03-24 15:12:00+08:00  0.0020      0.002      0.0167        0.119760
7 2021-03-24 15:14:00+08:00  0.0020      0.002      0.0167        0.119760
8 2021-03-24 15:15:00+08:00  0.0020      0.002      0.0167        0.119760```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

# Ensure full column string display (no truncation) for kpi_name when showing results
pd.set_option('display.max_colwidth', None)

tz = pytz.timezone('Asia/Shanghai')

# Reuse dockerA2_metric_aggr if available; otherwise recompute from df_metric and dockerA2_metric_rows
try:
    dockerA2_metric_aggr
except NameError:
    # ensure df_metric exists and timestamp is timezone-aware
    try:
        df_metric
    except NameError:
        df_metric = pd.read_csv("metric.csv")
        df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    if not pd.api.types.is_datetime64tz_dtype(df_metric['timestamp']):
        df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    start_ts = pd.Timestamp("2021-03-24 15:00:00").tz_localize(tz)
    end_ts   = pd.Timestamp("2021-03-24 15:30:00").tz_localize(tz)
    dockerA2_metric_rows = df_metric[(df_metric['cmdb_id']=='dockerA2') & (df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()
    dockerA2_metric_aggr = (
        dockerA2_metric_rows
        .groupby('kpi_name', as_index=False)['value']
        .agg(count='count', min_value='min', max_value='max', mean_value='mean')
        .sort_values('max_value', ascending=False)
        .reset_index(drop=True)
    )

# Prepare the top 20 rows sorted by max_value descending with full kpi_name strings
dockerA2_metric_aggr_full = dockerA2_metric_aggr.sort_values('max_value', ascending=False).head(20)[
    ['kpi_name', 'count', 'min_value', 'max_value', 'mean_value']
].reset_index(drop=True)

# Return the result (keeps variables in kernel for later inspection)
dockerA2_metric_aggr_full

```
Out[10]:
```
```python
summary = (
    "DockerA2 metric snapshot (2021-03-24 15:00–15:30 UTC+8) — key findings:\n"
    "- Top KPIs (by max value) are container-level metrics for container UATWKR06:\n"
    "  • MemLimit ≈ 3.07e9, observed as constant (count=3) — container memory limit ~3.07 GB.\n"
    "  • MemUsage ≈ 1.98e9 (mean ≈1.98e9, count=10) — container using ~1.98 GB.\n"
    "  • NetworkRxBytes mean ≈ 2.02e6 and NetworkTxBytes mean ≈ 3.56e5 (count=10 each) — notable network traffic.\n"
    "  • MemPercent ≈ 59% (count=5) and CpuPercent small (mean ≈0.624, max ≈0.82) — memory utilization is significant while CPU is low.\n"
    "- Many other container-related KPI groups are all zeros in this window (likely inactive or not reporting).\n"
    "Interpretation:\n"
    "- DockerA2 shows substantial memory usage (≈2 GB) on container UATWKR06 with ~59% memory occupancy and non-trivial network I/O, while CPU is low.\n"
    "- Combined with the earlier trace anomaly (dockerA2 trace.self.duration_p95 spike at 15:01), this suggests a container-level resource or network-related event affecting latency rather than CPU saturation.\n"
    "Recommended next steps:\n"
    "- Inspect container UATWKR06 around 15:00–15:08: container logs, OOM events, GC (if JVM), and orchestration events (restarts/limits).\n"
    "- Correlate memory and network spikes with trace p95 spikes and application logs to determine whether memory pressure, network bursts, or transient IO caused the latency.\n"
    "- If needed, examine host-level metrics (disk I/O, network interface) and pod/container config (limits/requests) for remediation."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name  count     min_value     max_value    mean_value
0         Container-DOCKER_CONTAINER_2c2336e2994f--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_MemLimit      3  3.072000e+09  3.072000e+09  3.072000e+09
1         Container-DOCKER_CONTAINER_2c2336e2994f--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_MemUsage     10  1.977958e+09  1.983508e+09  1.980701e+09
2   Container-DOCKER_CONTAINER_2c2336e2994f--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_NetworkRxBytes     10  1.285224e+06  3.197472e+06  2.015719e+06
3   Container-DOCKER_CONTAINER_2c2336e2994f--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_NetworkTxBytes     10  2.636090e+05  4.702310e+05  3.559610e+05
4       Container-DOCKER_CONTAINER_2c2336e2994f--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_MemPercent      5  5.990000e+01  5.990000e+01  5.990000e+01
5       Container-DOCKER_CONTAINER_2c2336e2994f--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_CpuPercent      5  4.000000e-01  8.200000e-01  6.240000e-01
6       Container-DOCKER_CONTAINER_3bf443a64876--bcou-role-st-uat-statefulset-1--bcou--UATWKR03_CpuPercent     13  0.000000e+00  0.000000e+00  0.000000e+00
7         Container-DOCKER_CONTAINER_3bf443a64876--bcou-role-st-uat-statefulset-1--bcou--UATWKR03_MemLimit      2  0.000000e+00  0.000000e+00  0.000000e+00
8       Container-DOCKER_CONTAINER_3bf443a64876--bcou-role-st-uat-statefulset-1--bcou--UATWKR03_MemPercent      4  0.000000e+00  0.000000e+00  0.000000e+00
9         Container-DOCKER_CONTAINER_3bf443a64876--bcou-role-st-uat-statefulset-1--bcou--UATWKR03_MemUsage      7  0.000000e+00  0.000000e+00  0.000000e+00
10  Container-DOCKER_CONTAINER_3bf443a64876--bcou-role-st-uat-statefulset-1--bcou--UATWKR03_NetworkTxBytes      6  0.000000e+00  0.000000e+00  0.000000e+00
11      Container-DOCKER_CONTAINER_69b53a78b2eb--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_CpuPercent      7  0.000000e+00  0.000000e+00  0.000000e+00
12      Container-DOCKER_CONTAINER_69b53a78b2eb--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_MemPercent      7  0.000000e+00  0.000000e+00  0.000000e+00
13        Container-DOCKER_CONTAINER_69b53a78b2eb--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_MemUsage      1  0.000000e+00  0.000000e+00  0.000000e+00
14  Container-DOCKER_CONTAINER_69b53a78b2eb--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_NetworkTxBytes      1  0.000000e+00  0.000000e+00  0.000000e+00
15      Container-DOCKER_CONTAINER_76d31070b844--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_CpuPercent      4  0.000000e+00  0.000000e+00  0.000000e+00
16        Container-DOCKER_CONTAINER_76d31070b844--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_MemLimit      3  0.000000e+00  0.000000e+00  0.000000e+00
17      Container-DOCKER_CONTAINER_76d31070b844--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_MemPercent     12  0.000000e+00  0.000000e+00  0.000000e+00
18        Container-DOCKER_CONTAINER_76d31070b844--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_MemUsage     18  0.000000e+00  0.000000e+00  0.000000e+00
19  Container-DOCKER_CONTAINER_76d31070b844--bcou-role-st-uat-statefulset-1--bcou--UATWKR06_NetworkTxBytes     16  0.000000e+00  0.000000e+00  0.000000e+00```
```